# Day 4: Story Weaver - Fine-Tuning DistilGPT-2

**Objective**: Fine-tune the DistilGPT-2 model with `wizard_of_oz_cleaned.txt` to generate more coherent, Oz-specific stories, reducing drift and repetition.

**Steps**:
1. Prepare the dataset from the cleaned text.
2. Set up and train the model.
3. Test the fine-tuned model with the choice system.
4. Save and document the results.

**Note**: Requires sufficient RAM (at least 8GB) and disk space (~500MB for training).

## preparing the dataset

In [6]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from torch.utils.data import Dataset
import os

# Define file paths
project_root = os.path.join("..")
cleaned_file = os.path.join(project_root, "data", "processed", "wizard_of_oz_cleaned.txt")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token  # Set pad token to eos token

# Custom dataset class to handle large files
class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, block_size=128):
        self.examples = []
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()  # Read as full text
            tokens = tokenizer.encode(text, add_special_tokens=True)  # Tokenize properly
            for i in range(0, len(tokens) - block_size + 1, block_size):
                chunk = tokens[i:i + block_size]
                self.examples.append(chunk)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return {"input_ids": self.examples[i], "labels": self.examples[i]}

# Prepare dataset
dataset = TextDataset(cleaned_file, tokenizer)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f"Dataset prepared with {len(dataset)} chunks of {dataset[0]['input_ids'][:5]}... (total tokens: {sum(len(x) for x in dataset.examples)})")

Token indices sequence length is longer than the specified maximum sequence length for this model (55956 > 1024). Running this sequence through the model will result in indexing errors


Dataset prepared with 437 chunks of [58, 21478, 44027, 60, 198]... (total tokens: 55936)


## Setup and training the model

In [7]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

# Load pre-trained model
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# Define training arguments with adjusted learning rate
training_args = TrainingArguments(
    output_dir=os.path.join(project_root, "models", "fine_tuned_distilgpt2"),
    overwrite_output_dir=True,
    num_train_epochs=5,  # Increased to 5 for better learning
    per_device_train_batch_size=2,
    learning_rate=2e-5,  # Lower learning rate for stability
    save_steps=500,
    save_total_limit=2,
    logging_dir=os.path.join(project_root, "logs"),
    logging_steps=100,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

# Train the model
trainer.train()
print("Model training completed!")

# Save the fine-tuned model
trainer.save_model()
print("Fine-tuned model saved to models/fine_tuned_distilgpt2/")

Step,Training Loss
100,3.428300
200,3.226500
300,2.981700
400,2.981100
500,2.861100
600,2.885700
700,2.799300
800,2.740600
900,2.814000
1000,2.736100


Model training completed!
Fine-tuned model saved to models/fine_tuned_distilgpt2/


## Test and fine tuned model

In [8]:
from transformers import pipeline

# Load fine-tuned model
fine_tuned_generator = pipeline('text-generation', model=os.path.join(project_root, "models", "fine_tuned_distilgpt2"), tokenizer="distilgpt2")

# Manually set a test choice
selected_choice = 1  # Change to 2 for river bed

# Test with a stronger Oz-specific prompt
if selected_choice == 1:
    test_prompt = "Dorothy chose to follow the golden road to the old temple in the land of Oz, a land of magic and wonder. The temple stood ancient and mysterious..."
elif selected_choice == 2:
    test_prompt = "Dorothy chose to follow the golden road to the river bed where children lived in the land of Oz, a land of magic and wonder. The river sparkled with magical light..."

# Generate story with improved decoding
story = fine_tuned_generator(test_prompt, max_length=250, num_return_sequences=1, temperature=0.7, 
                            no_repeat_ngram_size=2, truncation=True)
generated_story = story[0]['generated_text']
# Ensure proper decoding with spaces
generated_story = " ".join(generated_story.split())  # Add spaces between tokens
print("\nGenerated Story with Fine-Tuned Model:")
print(generated_story)

Device set to use mps:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Generated Story with Fine-Tuned Model:
Dorothy chose to follow the golden road to the old temple in the land of Oz, a land of magic and wonder. The temple stood ancient and mysterious... The Lion was a beautiful tree, and she was so beautiful that Dorothy could not see her eyes. Dorian Woodman had never seen the tree again, but the Scarecrow had been one of the greatest spirits ever seen. He was the King of The Monkeys, and he was always in great fear. For he had seen many strange creatures, some of them small and quite big, that were so big that they could only see them in their arms. So Dorothy walked by and asked who the Lion was, telling the answer: “The Witch of Norway.” For he knew that the Witch was a Wizard, who lived in her land. But he did not know who she was; and he kept away from her until she came to him. When she saw what she had, she thought she would tell him so. After she made a deep sigh, he called, “Come on,’― said to his friend, standing beside him. ‘


## Initilize story state

In [10]:
# Initialize story state based on the selected choice
story_state = {
    "current_scene": "temple" if selected_choice == 1 else "river_bed",
    "choices_made": [choices["crossroads"][selected_choice] if 'choices' in locals() else "manual_test"],
    "last_prompt": ""
}

print("Initialized story state:", story_state)

Initialized story state: {'current_scene': 'temple', 'choices_made': ['manual_test'], 'last_prompt': ''}


## Saving the story

In [11]:
# Save story to output
output_file = os.path.join(project_root, "output", f"generated_story_fine_tuned_{story_state['current_scene']}_2025-04-16.txt")
os.makedirs(os.path.join(project_root, "output"), exist_ok=True)
with open(output_file, 'w', encoding='utf-8') as file:
    file.write(generated_story)
print(f"Story saved to: {output_file}")

Story saved to: ../output/generated_story_fine_tuned_temple_2025-04-16.txt


## Next Steps for Day 4

**Done**:
- Fine-tuned DistilGPT-2 with `wizard_of_oz_cleaned.txt`.
- Tested the model with a sample prompt.
- Saved the generated story.

**To Do**:
- Compare fine-tuned vs. pre-trained outputs.
- Expand the story plan with new scenes.
- Prepare for Day 5: Add a simple UI (e.g., Flask).

**Action**: Commit changes to GitHub and update `.gitignore` with `models/` and `logs/`.